Parse all dates, normalise them with AI

For external API
Use external API to get lat and long for the place where the event happend. That way all the locations will be normalized 

Fix Sex Column - 11 Genders! Its 2026, all right
Fix Age Column - 251 unique value, WTF!
Fix Country - Maybe directly with API - 252 value, a bit too much 
Fix Type - Some are dublicated

Fix all unknown, sometimes they are Unknown, sometimes ? and sometimes Nan

Fix Date

In [40]:
%pip install dateparser

Note: you may need to restart the kernel to use updated packages.


In [41]:
import pandas as pd
import dateparser 

pd.set_option("display.max_columns", None)

In [42]:
data = pd.read_excel("GSAF5.xls")
# keep original text for reference
data["Date_orig"] = data["Date"]

import re
from datetime import datetime, timedelta


def parse_date_info(x, year=None):
    """Parse a raw date string and return a dict with normalization

    ``year`` is an optional secondary year value (from the ``Year`` column).
    When the text lacks a four‑digit year, this value is used to fill in the
    missing component (e.g. "29th January" with Year=1999).

    - ``normalized``: a datetime representing the best guess for the event
      (used to populate ``Date`` column later).
    - ``is_exact_date``: ``True`` only if day, month and year are present in
      the original string.
    - ``date_from`` / ``date_to``: for ranges/periods.
    - ``date_extra``: any removed/annotated text ("reported", "before ...",
      etc.) or a note for unparsed values.
    """
    info = {
        "normalized": None,
        "is_exact_date": False,
        "date_from": None,
        "date_to": None,
        "date_extra": None,
    }
    if pd.isna(x):
        return info
    s = str(x).strip()

    # pull year from separate column if available; will be used later
    orig_year = None
    if year is not None and not pd.isna(year):
        try:
            orig_year = int(year)
        except Exception:
            orig_year = None

    # quick negative cases
    if re.match(r'(?i)^(no\s*date|none|nan)\b', s):
        info["date_extra"] = "no date"
        return info

    # housekeeping / normalization of input text ------------------------------------------------
    # drop leading and trailing junk
    s = s.rstrip(". ")
    # drop a leading "Reported" but record that fact
    if s.lower().startswith("reported"):
        s = s[len("reported"):].strip()
        info["date_extra"] = "reported"

    # remove stray letters after a valid date ("19-Jul-2007.b")
    s = re.sub(r"(\d{1,2}-[A-Za-z]{3}-\d{2,4})[\.][A-Za-z]$", r"\1", s)

    # convert various separators to plain spaces for easier matching
    s = s.replace("–", "-")
    s = s.replace("/", " ")

    # handle obvious OCR/truncation issues
    # e.g. "190Feb-2010" → "19 Feb-2010"
    s = re.sub(r"(?i)\b(\d{3})([A-Za-z]+)\b", lambda m: m.group(1)[:2] + " " + m.group(2), s)

    # mid/early/late prefixes that dateparser can't handle
    m_phase = re.match(r"(?i)^(early|mid|late)\s+([A-Za-z]+)[\s\-](\d{4})", s)
    if m_phase:
        phase = m_phase.group(1).lower()
        month_name = m_phase.group(2)
        year = int(m_phase.group(3))
        # try parsing month name generically
        try:
            tmp = dateparser.parse(f"1 {month_name}")
            month = tmp.month if tmp else 1
        except Exception:
            month = 1
        if phase == "early":
            day = 5
        elif phase == "mid":
            day = 15
        else:
            # late
            # pick 28 to account for February or months with 30/31
            day = 28
        info["normalized"] = datetime(year, month, day)
        info["is_exact_date"] = False
        info["date_extra"] = phase
        return info

    # "Late Jul-2008" → we'll just drop the "Late" and later assign day=28
    if re.match(r"(?i)^late\b", s):
        s = re.sub(r"(?i)^late\s+", "", s)
        # treat in date_extra so we know it was approximate
        info["date_extra"] = ((info.get("date_extra") or "") + " late").strip()

    # seasons that dateparser doesn't handle directly
    season_map = {
        "spring": (4, 15),
        "summer": (7, 15),
        "fall": (10, 15),
        "autumn": (10, 15),
        "winter": (1, 15),
    }
    for season, (month, day) in season_map.items():
        if re.match(rf"(?i)^{season}[\s\-](\d{{4}})", s):
            year = int(re.search(r"(\d{4})", s).group(1))
            info["normalized"] = datetime(year, month, day)
            info["is_exact_date"] = False
            info["date_extra"] = season
            return info

    # explicit periods like ``1845-1853`` or ``1845 – 1853``
    m_range = re.match(r'^(?P<start>\d{4})\s*[-–]\s*(?P<end>\d{4})$', s)
    if m_range:
        y1 = int(m_range.group("start"))
        y2 = int(m_range.group("end"))
        dt1 = datetime(y1, 1, 1)
        dt2 = datetime(y2, 12, 31)
        info["date_from"] = dt1
        info["date_to"] = dt2
        info["normalized"] = dt1 + (dt2 - dt1) / 2
        info["is_exact_date"] = False
        info["date_extra"] = f"period {y1}-{y2}"
        return info

    # "Between 1918 & 1939" style ranges
    m_between = re.match(r'(?i)^between\s*(?P<start>\d{4})\s*[&and]+\s*(?P<end>\d{4})', s)
    if m_between:
        y1 = int(m_between.group("start"))
        y2 = int(m_between.group("end"))
        dt1 = datetime(y1, 1, 1)
        dt2 = datetime(y2, 12, 31)
        info["date_from"] = dt1
        info["date_to"] = dt2
        info["normalized"] = dt1 + (dt2 - dt1) / 2
        info["is_exact_date"] = False
        info["date_extra"] = f"between {y1}-{y2}"
        return info

    # "Before <date>" case (year or full date)
    m_before = re.match(r'(?i)^before\s*(.+)', s)
    if m_before:
        stripped = m_before.group(1).strip()
        # try to parse the stripped portion normally
        dt = dateparser.parse(stripped)
        if dt:
            info["date_to"] = dt
            # choose a normalized point halfway between start of era and dt
            info["normalized"] = dt - timedelta(days=30)
            info["is_exact_date"] = False
            info["date_extra"] = f"before {dt.date()}"
            return info
        else:
            # fallback to only capturing year
            year_match = re.match(r"(\d{4})", stripped)
            if year_match:
                year = int(year_match.group(1))
                info["date_to"] = datetime(year, 1, 1)
                info["normalized"] = datetime(year, 6, 30)
                info["is_exact_date"] = False
                info["date_extra"] = f"before {year}"
                return info

    # final fallback: let dateparser try to do the heavy lifting
    dt = dateparser.parse(s)
    if dt:
        # if year absent from string but available separately, inject it
        if orig_year and not re.search(r'\b\d{4}\b', s):
            try:
                dt = dt.replace(year=orig_year)
            except Exception:
                pass
        info["normalized"] = dt
        # crude check for presence of day and month in original text
        has_day = bool(re.search(r'\b([0-3]?\d)\b', s)) and not bool(
            re.match(r'^\d{4}$', s)
        )
        has_month = bool(
            re.search(r'(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec|\d{1,2}[\s\-]\d{1,2})', s, re.I)
        )
        info["is_exact_date"] = has_day and has_month

        # after parsing, check for weekend mention – shift back a day
        if re.search(r'weekend', s, re.I) and info["normalized"]:
            info["normalized"] = info["normalized"] - timedelta(days=1)
            info["date_extra"] = (info.get("date_extra") or "") + " weekend"
        # handle approximations
        if info["is_exact_date"] is False:
            if "late" in (info.get("date_extra") or ""):
                info["normalized"] = info["normalized"].replace(day=28)
            elif "early" in (info.get("date_extra") or ""):
                info["normalized"] = info["normalized"].replace(day=5)
            elif "mid" in (info.get("date_extra") or ""):
                info["normalized"] = info["normalized"].replace(day=15)
        return info

    # nothing could be parsed
    info["date_extra"] = "unparsed"
    return info


# apply parser row-wise so the Year column can be used when the string lacks a year
parsed = data.apply(lambda r: parse_date_info(r["Date_orig"], r.get("Year")), axis=1)
parsed_df = pd.DataFrame(parsed.tolist(), index=data.index)

# merge new columns into the main frame
data = pd.concat([data, parsed_df], axis=1)

# post-processing fixes --------------------------------------------------
# 1. rows where year was truncated (e.g. "10-Jul-202"); borrow year from next
mask_short_year = data["Date_orig"].astype(str).str.match(r"\d{1,2}-[A-Za-z]{3}-\d{1,3}$")
for idx in data[mask_short_year & data["normalized"].isna()].index:
    nxt = idx + 1
    if nxt in data.index and pd.notna(data.loc[nxt, "normalized"]):
        year = data.loc[nxt, "normalized"].year
        orig = data.loc[idx, "Date_orig"]
        corrected = re.sub(r"(\d{1,2}-[A-Za-z]{3}-)(\d{1,3})$", lambda m: m.group(1) + str(year), orig)
        repair = parse_date_info(corrected)
        for key, val in repair.items():
            data.at[idx, key] = val
        if repair["normalized"]:
            data.at[idx, "Date"] = repair["normalized"].strftime("%Y-%m-%d")

# 2. ensure "date_to" is set for before cases even if normalized was filled earlier
before_mask = data["Date_orig"].astype(str).str.lower().str.startswith("before")
for idx in data[before_mask].index:
    if pd.isna(data.at[idx, "date_to"]) and pd.notna(data.at[idx, "normalized"]):
        data.at[idx, "date_to"] = data.at[idx, "normalized"]

# overwrite the original "Date" column with ISO repr of normalized value
# (it may already have been set by first pass, but redo to catch repairs)
data["Date"] = data["normalized"].dt.strftime("%Y-%m-%d")

# capture rows that still failed to produce a normalized datetime
suspicious_dates_idx = data[data["normalized"].isna()].index
pd.DataFrame(suspicious_dates_idx).to_csv("idx.csv")


In [43]:
data

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,Injury,Fatal Y/N,Time,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22,Date_orig,normalized,is_exact_date,date_from,date_to,date_extra
0,2026-01-29,2026.0,Unprovoked,Brazil,Recife,Del Chifre Beach in Olinda,Swimming,Deivson Rocha Dantas,M,13,Right thigh and lower leg stripped of flesh,Y,?,Unknown bull and tiger sharks frequent the area,Kevin McMurray Trackingsharks.com: TV Globo: P...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29th January,2026-01-29,False,NaT,NaT,NaN
1,2026-01-29,2026.0,Unprovoked,Australia,NSW,Angels Beach East Ballina,Surfing,Unnamed man,M,?,No injury shark knocked man of his board,N,1100hrs,Unknown,Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29th January,2026-01-29,False,NaT,NaT,NaN
2,2026-01-24,2026.0,Unprovoked,Australia,Tasmania,Cooee Beach west of Burnie,Swimming,Megan Stokes,F,?,Puncture wounds to right knee,N,1815hrs,1.7m Seven Gill shark,Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24th January,2026-01-24,False,NaT,NaT,NaN
3,2026-01-20,2026.0,Unprovoked,Australia,NSW,Point Plomber North of Port Macquarie,Surfing,Paul Zvirdinas,M,39,Minor cuts and abrasions,N,0830hrs,Bull shark,Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20th January,2026-01-20,False,NaT,NaT,NaN
4,2026-01-19,2026.0,Unprovoked,Australia,NSW,Dee Why,Surfing,Unknown,M,11,None reported damage to board,N,1145hrs,Bull shark,Andy Currie,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19th January,2026-01-19,False,NaT,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7069,1903-01-28,0.0,Unprovoked,AUSTRALIA,Western Australia,Roebuck Bay,Diving,male,M,NaN,FATAL,Y,NaN,NaN,"H. Taunton; N. Bartlett, p. 234",ND-0005-RoebuckBay.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0005,ND.0005,6.0,NaN,NaN,Before 1903,1903-01-28,False,NaT,1903-02-27,before 1903-02-27
7070,1903-01-28,0.0,Unprovoked,AUSTRALIA,Western Australia,NaN,Pearl diving,Ahmun,M,NaN,FATAL,Y,NaN,NaN,"H. Taunton; N. Bartlett, pp. 233-234",ND-0004-Ahmun.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0004,ND.0004,5.0,NaN,NaN,Before 1903,1903-01-28,False,NaT,1903-02-27,before 1903-02-27
7071,1903-01-01,0.0,Unprovoked,USA,North Carolina,Ocracoke Inlet,Swimming,Coast Guard personnel,M,NaN,FATAL,Y,NaN,NaN,"F. Schwartz, p.23; C. Creswell, GSAF",ND-0003-Ocracoke_1900-1905.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0003,ND.0003,4.0,NaN,NaN,1900-1905,1903-01-01,False,1900-01-01,1905-12-31,period 1900-1905
7072,1886-07-02,0.0,Unprovoked,PANAMA,NaN,"Panama Bay 8ºN, 79ºW",NaN,Jules Patterson,M,NaN,FATAL,Y,NaN,NaN,"The Sun, 10/20/1938",ND-0002-JulesPatterson.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0002,ND.0002,3.0,NaN,NaN,1883-1889,1886-07-02,False,1883-01-01,1889-12-31,period 1883-1889


In [44]:
data[data["normalized"].isna()]

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,Injury,Fatal Y/N,Time,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22,Date_orig,normalized,is_exact_date,date_from,date_to,date_extra
2398,NaN,2004.0,Watercraft,AUSTRALIA,Western Australia,"7 km off Trigg surf beach, Perth",Fishing,"boat, occupants: Mike Taylor & his son, Jack,...",NaN,NaN,"No injury to occupants; shark mouthed motor, s...",N,NaN,4 m [13'] white shark,"Channel 9; Northern Territory News, 7/20/2004,...",2004.07.19.R-boat.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2004.07.19.R,2004.07.19.R,4672.0,NaN,NaN,"19-Jul-2004 Reported to have happened ""on th...",NaT,False,NaT,NaT,unparsed
3203,NaN,1994.0,Unprovoked,HONG KONG,NaN,NaN,Playing volleyball with friends,female,F,NaN,Both legs lacerated,N,NaN,Tiger shark said to be 5 to 7 m [16.5' to 23'],J. Wong,1994.00.00.b-NV-HongKong.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,1994.00.00.b,1994.00.00.b,3867.0,NaN,NaN,Last incident of 1994 in Hong Kong,NaT,False,NaT,NaT,unparsed
3258,NaN,1993.0,Unprovoked,SOMALIA,Banaadir Region,Mogadishu,NaN,several Somali children,NaN,NaN,FATAL,Y,NaN,NaN,"Orlando Sentinel, 112/1/1993, p.A9",1993.00.00.c - Somali children.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,1993.00.00.c,1993.00.00.c,3812.0,NaN,NaN,Between May & Nov-1993,NaT,False,NaT,NaT,unparsed
3796,NaN,1981.0,Unprovoked,GREECE,Pagasitikos Gulf,NaN,"Free diving / spearfishing,",an Austrian tourist,M,NaN,Minor injury,N,NaN,NaN,M. Bardanis,1981.00.00.a-Austrian-diver.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,1981.00.00.a,1981.00.00.a,3274.0,NaN,NaN,Summer of 1981,NaT,False,NaT,NaT,unparsed
4281,NaN,1967.0,Provoked,AUSTRALIA,South Australia,Outer Harbor,NaN,Rodney Baim,M,30,Thigh abraded & lacerated Recorded as PROVOKED...,N,Morning,Bronze whaler shark,"J. Green, p.36; H. D. Baldridge (1994) SAF Cas...",1967.05.13-NV-Baim.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,1967.05.13,1967.05.13,2789.0,NaN,NaN,13 or 30-May-1967,NaT,False,NaT,NaT,unparsed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7048,NaN,0.0,Unprovoked,BELIZE,NaN,NaN,Standing,a servant,M,16,FATAL,Y,NaN,12' tiger shark,Mitchell-Hedges,ND-0026-Belize.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0026,ND.0026,27.0,NaN,NaN,Early 1930s,NaT,False,NaT,NaT,unparsed
7051,NaN,0.0,Unprovoked,SOUTH AFRICA,Western Cape Province,Arniston,Wading,Madelaine Dalton,F,NaN,Ankle bitten,N,NaN,NaN,"L. Green in Tavern of the Seas, p.182",ND-0023-Dalton.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0023,ND.0023,24.0,NaN,NaN,No date,NaT,False,NaT,NaT,no date
7052,NaN,0.0,Unprovoked,AUSTRALIA,NaN,NaN,Pearl diving,Jaringoorli,M,NaN,Lacerations to thigh,N,NaN,NaN,"Adelaide Advertiser, 1/11/1940",ND-0022-Jaringoorli.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0022,ND.0022,23.0,NaN,NaN,No date,NaT,False,NaT,NaT,no date
7053,NaN,0.0,Unprovoked,SOUTH AFRICA,KwaZulu-Natal,Durban,Swimming in pool formed by construction of a w...,Indian boy,M,NaN,"FATAL, leg severed",Y,NaN,NaN,"L. Green in South African Beachcomber, p.97",ND-0021-DurbanIndianBoy.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0021,ND.0021,22.0,NaN,NaN,No date,NaT,False,NaT,NaT,no date


In [ ]:
unprocessed_data = pd.read_excel("GSAF5.xls")
unprocessed_data["Date"] = unprocessed_data["Date"].astype(str)

sorted_unprocessed = unprocessed_data.sort_values(by="Date")

sorted_processed = data.sort_values(by="Date")

print(sorted_unprocessed.shape)
print(sorted_processed.shape)



(7074, 23)
(7074, 29)
